# 09 · LLM Planner — ReAct + 提示工程 + JSON-Schema 绑定 (维度 1.4/2.1/2.2/2.3)

铁律: **LLM 提议、引擎裁决**。LLM 只抽字段/选技能/起草回复; 数字·工艺冲突·状态由确定性引擎+护栏+状态机定。
LLM 离线 → 显式 MOCK 降级, 回退确定性实现, 不阻断。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.llm_planner import LLMPlanner, _extract_json, _validate
# 纯函数: 稳健 JSON 抽取 + schema 校验 (离线可验证)
print('fenced:', _extract_json('```json\n{"material":"6061"}\n```'))
print('noisy :', _extract_json('here: {"quantity":50} thanks'))
print('schema err:', _validate({'quantity':5}, {'type':'object','required':['material']}))

In [ ]:
p = LLMPlanner()
print('LLM online:', p.online(), '| source:', p.source_label())
r = p.extract_rfq('Please quote 50 pcs 6061 aluminum brackets, anodizing, tolerance IT7.')
print('extract_rfq ok=', r.get('ok'), '| mock=', r.get('_mock'), '| data=', r.get('data'))

## ReAct 循环: LLM 选技能 (受 allow-list 约束), 离线走确定性回退序列

In [ ]:
allowed = {'rfq-extraction','dfm-conflict','cnc-quote','verification','finish'}
executed = []
out = p.react_loop('STRUCTURING', lambda h: 'material=6061 surface=阳极氧化 qty=50',
                   lambda tool, args: executed.append(tool) or {'ok': True},
                   allowed, max_steps=6,
                   fallback_sequence=['rfq-extraction','dfm-conflict','cnc-quote','verification'])
print('mock=', out['_mock'], '| executed=', executed)
print('trace steps=', len(out['trace']))

## 端到端: use_llm=True 时 LLM 抽取补全 (引擎仍裁决)

In [ ]:
ctrl = build_controller()
r = ctrl.run(email_text='need 50 pcs 6061 anodizing IT7', customer={'name':'LLM Demo'}, use_llm=True)
print('state=', r['state'], '| llm_planner=', r['llm_planner'])
print('注: LLM 离线时 used=False, 走确定性正则抽取, 结果不变 (守铁律)')
ctrl.crm.close()